# Модуль 5: LoRA-донавчання Qwen3-0.6B у Google Colab

1. Меню **Середовище виконання → Змінити тип середовища → GPU T4** → Зберегти.
2. Виконуйте комірки по черзі (Shift+Enter). Загалом ≈ 15–20 хв.
3. В останній комірці завантажиться `qwen3-lora.zip` — розпакуйте його в папку `models\` програми на своєму ПК.

У Colab потрапляють лише синтетичні навчальні приклади з архіву програми — нічого іншого не завантажуйте.

## Крок 1. Код програми
Оберіть **один** варіант: A — завантажити архів `TL-Lab.zip` з ПК (працює завжди); B — з GitHub (коли репозиторій буде викладено).

In [ ]:
# Варіант A: з'явиться кнопка «Вибрати файли» — оберіть TL-Lab.zip
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd /content/TL-Lab/app
print(os.listdir('.'))

In [ ]:
# Варіант B (пізніше): з GitHub
# !git clone https://github.com/Dicpano/Praktika2026.git
# %cd Praktika2026/TL-Lab/app

## Крок 2. Бібліотеки та перевірка GPU

In [ ]:
!pip -q install 'transformers>=4.51,<5' 'peft>=0.13' accelerate sentence-transformers huggingface_hub
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕМАЄ — увімкніть T4 і перезапустіть')

## Крок 3. Готова модель Qwen3-0.6B (≈1,5 ГБ)

In [ ]:
from scripts.download_models import m5
m5()

## Крок 4. Донавчання LoRA (≈5–10 хв)

In [ ]:
from modules import m5_llm as m
from core.jobs import NullJob
hist = m.train_lora(epochs=3, lr=2e-4, r=8, alpha=16, job=NullJob(verbose=True))

## Крок 5. Порівняння до / після на 6 тестових текстах

In [ ]:
import json
r = m.run_experiment(modes=('zero', 'few', 'lora'), job=NullJob(verbose=True))
print(json.dumps(r['metrics']['by_mode'], ensure_ascii=False, indent=1))
for md_, outs in r['outputs'].items():
    print('=====', md_); print(outs[0])

## Крок 6. Завантажити адаптер на ПК

In [ ]:
!cd models && zip -qr ../qwen3-lora.zip qwen3-lora
from google.colab import files
files.download('qwen3-lora.zip')